# Lab 19 — Choosing a Framing, Neural Networks, and Comparing Algorithms Head-to-Head

In this lab you will decide whether a problem should be framed as regression or classification and see how that choice changes both the model and the evaluation metric, train a small feedforward neural network and see how much added capacity changes its accuracy on real handwritten digits, and then run a full cross-validated benchmark — including a neural network — to pick the best classifier for a new dataset and evaluate it honestly on a held-out test set.

**Concepts covered:** Regression vs. classification problem framing, feedforward neural network architecture and capacity (depth vs. width), feature scaling for neural networks, cross-validated model comparison across classical and neural network algorithms, and the No Free Lunch theorem in practice.

**Reference working sessions:**
- `working-sessions/supervised/15_regression_vs_classification_selection.ipynb`
- `working-sessions/supervised/16_neural_networks_intro.ipynb`
- `working-sessions/supervised/17_model_comparison.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_digits, fetch_california_housing
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, roc_auc_score, accuracy_score

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_heart_disease,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'A',
    'q3': 'B',
    'q4': 'A',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — A regional clinic wants to know how many days a patient will
# likely stay in the hospital, so they can plan staffing and beds. Per
# the decision framework in
# working-sessions/supervised/15_regression_vs_classification_selection.ipynb,
# should this be framed as regression or classification?
#
#   A) Classification, because health-related targets are always
#      categorical
#   B) Regression, because the target is a continuous quantity (days),
#      and the clinic needs the actual magnitude, not just a yes/no or
#      category label
#   C) Either framing is equally correct, and the choice never affects
#      which evaluation metric you should use
#   D) Classification, because you can always bucket the days into
#      categories after training, so the original framing doesn't matter

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/supervised/" \
    "15_regression_vs_classification_selection.ipynb and its decision guide."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — In the architecture widget in
# working-sessions/supervised/16_neural_networks_intro.ipynb, going from
# 1 hidden layer of 32 neurons to 2 hidden layers of 128 neurons
# multiplies the parameter count by roughly 4.6x, not 2x. What causes
# this non-linear growth?
#
#   A) Every layer transition is a full weight matrix connecting every
#      neuron in one layer to every neuron in the next, so adding a
#      layer multiplies connections — it doesn't just add a few more
#      neurons
#   B) Plotly rounds the displayed number up for readability
#   C) sklearn silently doubles the learning rate for every additional
#      hidden layer
#   D) Bias terms are counted twice for every additional layer

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/supervised/16_neural_networks_intro.ipynb " \
    "and the architecture widget's \"What's happening?\" section."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — Before training a neural network on the pixel values of an
# image (each pixel 0-255), why is it important to scale them down to
# a 0-1 range first, as covered in
# working-sessions/supervised/16_neural_networks_intro.ipynb?
#
#   A) sklearn's MLPClassifier automatically scales inputs internally,
#      so this step is purely a style preference
#   B) Neural networks are sensitive to the scale of their inputs;
#      unscaled, large-magnitude pixel values can slow down or
#      destabilize gradient-based training
#   C) Scaling changes how many parameters the network needs to learn
#   D) Scaling is required only for the output layer's activation
#      function, not the input pixels

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/supervised/16_neural_networks_intro.ipynb " \
    "and the \"Notice\" bullets in the real-world example section."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — In working-sessions/supervised/17_model_comparison.ipynb, no
# single algorithm wins on every metric and every dataset benchmarked.
# What idea does this best illustrate?
#
#   A) The No Free Lunch theorem: every algorithm's assumptions match
#      some datasets better than others, so the right choice depends on
#      your data and constraints, not a universal ranking
#   B) Accuracy is always the correct metric to optimize, regardless of
#      the business problem
#   C) More complex algorithms always outperform simpler ones, given the
#      same training data
#   D) Once you use cross-validation, the specific dataset you're
#      working with stops mattering

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/supervised/17_model_comparison.ipynb " \
    "and the \"Why this matters for data science\" section."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Choosing regression vs. classification for a real target"),
    (q2_answer, _ak['q2'], "Q2: Why parameter count grows faster than linearly with depth"),
    (q3_answer, _ak['q3'], "Q3: Why neural networks need scaled inputs"),
    (q4_answer, _ak['q4'], "Q4: What the No Free Lunch theorem says about algorithm choice"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below frame the same dataset two ways, compare a small neural network against a larger one on real digits, and run a quick cross-validated comparison of classical classifiers.

### B1 — Framing the same data two ways

Using the California Housing dataset, fit a regression model to predict the exact price, and a classification model to predict whether a district is above the median price. Compare their metrics.

In [ ]:
# B1 — Regression framing vs. classification framing on the same data
housing_X, housing_y_cont = fetch_california_housing(return_X_y=True, as_frame=True)
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    housing_X, housing_y_cont, test_size=0.2, random_state=42
)

reg_model = ___()   # YOUR CODE — the linear regression model
reg_model.fit(___, ___)   # YOUR CODE — the training features; the training continuous target
reg_r2 = r2_score(___, reg_model.predict(Xh_test))   # YOUR CODE — the true continuous test target

threshold = housing_y_cont.median()
y_bin = (housing_y_cont > threshold).astype(int)
Xh_train2, Xh_test2, yb_train, yb_test = train_test_split(
    housing_X, y_bin, test_size=0.2, random_state=42
)
clf_model = LogisticRegression(max_iter=1000, random_state=42)
clf_model.fit(___, ___)   # YOUR CODE — the training features; the training binary target
clf_auc = roc_auc_score(yb_test, clf_model.predict_proba(___)[:, 1])   # YOUR CODE — the held-out classification test features

print(f"Regression framing — R²: {reg_r2:.3f}")
print(f"Classification framing — AUC: {clf_auc:.3f}")

# --- checks ---
assert reg_r2 > 0.4, "Regression R² should be a plausible score for this dataset"
assert clf_auc > 0.7, "Classification AUC should be well above chance (0.5)"
print("✓ B1 complete!")

### B2 — Neural network capacity

Fit a small neural network and a larger one on real handwritten digits, and compare their accuracy.

In [ ]:
# B2 — Small vs. large neural network on real handwritten digits
digits_X, digits_y = load_digits(return_X_y=True)
digits_X_scaled = digits_X / 16.0  # raw pixel values are 0-16 in this dataset

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits_X_scaled, digits_y, test_size=0.2, random_state=42, stratify=digits_y
)

small_nn = ___(hidden_layer_sizes=(4,), max_iter=300, random_state=42)   # YOUR CODE — the sklearn neural network classifier used throughout this notebook
small_nn.fit(___, ___)   # YOUR CODE — the training features; the training labels
small_nn_acc = accuracy_score(yd_test, small_nn.predict(Xd_test))

large_nn = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=300, random_state=42)
large_nn.fit(Xd_train, yd_train)
large_nn_acc = accuracy_score(___, large_nn.predict(___))   # YOUR CODE — the true test labels; the test features

print(f"Small network (1 layer x 4 neurons):   accuracy = {small_nn_acc:.3f}")
print(f"Large network (2 layers x 64 neurons): accuracy = {large_nn_acc:.3f}")

# --- checks ---
assert small_nn_acc > 0.7, "Even the small network should score well above chance"
assert large_nn_acc > small_nn_acc, "The larger network should outperform the small one on this task"
print("✓ B2 complete!")

### B3 — Quick model comparison via cross-validation

Compare three classical classifiers on the Heart Disease dataset using 5-fold cross-validation, and identify the winner by mean accuracy.

In [ ]:
# B3 — Cross-validated comparison of three classical classifiers
Xc, yc = load_heart_disease()

models_b3 = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

cv_scores = {}
for name, model in models_b3.items():
    scores = cross_val_score(___, Xc, yc, cv=5, scoring='accuracy')   # YOUR CODE — the candidate model for this iteration of the loop
    cv_scores[name] = scores.mean()
    print(f"{name}: mean CV accuracy = {scores.mean():.3f}")

best_name = max(cv_scores, key=cv_scores.get)
print(f"\nBest by cross-validated accuracy: {best_name}")

# --- checks ---
assert len(cv_scores) == 3
assert all(v > 0.6 for v in cv_scores.values()), \
    "Every candidate should score well above chance on this dataset"
print("✓ B3 complete!")

---
## Section C — Applied Problem

A hospital wants to know which algorithm to trust for diagnosing heart disease — including whether a neural network is worth the extra complexity. The framing is already decided: this is a yes/no diagnosis, so it's classification. Benchmark Logistic Regression, Random Forest, and a neural network via cross-validation on the training set only, pick the winner by mean CV ROC-AUC, and evaluate it exactly once on the held-out test set.

In [ ]:
# Section C — Cross-validated benchmark including a neural network

# --- Step 1: Load and split ---
Xc, yc = load_heart_disease()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    ___, ___, test_size=___, random_state=___, stratify=___
    # YOUR CODE — the features and target; hold out 20% of the data with a
    # fixed seed for reproducibility, preserving the class balance
)

# Neural networks and logistic regression need scaled features; the
# tree-based Random Forest doesn't.
scaler = StandardScaler()
Xc_train_scaled = scaler.fit_transform(Xc_train)
Xc_test_scaled = scaler.transform(Xc_test)

# --- Step 2: Benchmark candidates via cross-validation on the training set only ---
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Neural Network": ___(hidden_layer_sizes=(64, 32), max_iter=500,
                          random_state=42, early_stopping=True),
    # YOUR CODE — the sklearn neural network classifier used throughout this notebook
}
NEEDS_SCALE = ("Logistic Regression", "Neural Network")

cv_results = {}
for name, model in candidates.items():
    X_use = Xc_train_scaled if name in NEEDS_SCALE else Xc_train
    scores = cross_val_score(model, ___, yc_train, cv=5, scoring='roc_auc')
    # YOUR CODE — the scaled-or-unscaled training features selected above for this candidate
    cv_results[name] = scores.mean()
    print(f"{name}: mean CV ROC-AUC = {scores.mean():.3f}")

# --- Step 3: Pick the winner, fit on the full training set, evaluate once ---
best_name = max(cv_results, key=cv_results.get)
best_model = candidates[best_name]
X_use_train = Xc_train_scaled if best_name in NEEDS_SCALE else Xc_train
X_use_test = Xc_test_scaled if best_name in NEEDS_SCALE else Xc_test
best_model.___(X_use_train, yc_train)   # YOUR CODE — the method that trains a model on data

final_auc = roc_auc_score(yc_test, best_model.predict_proba(___)[:, 1])
# YOUR CODE — the held-out test features matching the winner's scaling
print(f"\nWinner: {best_name} (test ROC-AUC = {final_auc:.3f})")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=list(cv_results.keys()), y=list(cv_results.values()),
            color=PALETTE["primary"], ax=ax)
ax.set_ylabel("Mean CV ROC-AUC")
ax.set_title("Model Comparison: Heart Disease (incl. Neural Network)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# --- checks ---
assert final_auc > 0.75, \
    "The winning model's held-out test ROC-AUC should be a plausible score, well above chance"
assert len(cv_results) == 3
print("✓ Section C complete!")
print(f"  Best candidate by cross-validation: {best_name}")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

You're now asked to predict the exact probability a patient has heart disease (a number between 0 and 1) rather than just a yes/no flag. Does this change the framing from classification to regression? Justify your answer using the decision criteria in `working-sessions/supervised/15_regression_vs_classification_selection.ipynb`.

*Your response here...*

**Question D2**

In Section C, the Neural Network scored the lowest mean CV ROC-AUC of the three candidates on the Heart Disease dataset. Using the "When NOT to use it" table in `working-sessions/supervised/16_neural_networks_intro.ipynb`, explain why a neural network might underperform simpler models here, even though it is a more powerful, more complex algorithm.

*Your response here...*

**Question D3**

Suppose the hospital later tells you they must be able to explain every individual prediction to a regulator. Using the interpretability-complexity spectrum table in `working-sessions/supervised/17_model_comparison.ipynb`, would you recommend keeping Section C's winning model, or switching to a different one? Justify your choice.

*Your response here...*